# Train Tabular + CLIP (Time-Based)

Основной multimodal ноутбук под новую схему данных.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor

PROJECT_ROOT = Path('/Users/zhasik/Desktop/krisha')
PARQUET_PATH = PROJECT_ROOT / 'data/index/index.parquet'
PROC_DIR = PROJECT_ROOT / 'data/processed'
OUT_DIR = PROJECT_ROOT / 'models'
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(PARQUET_PATH)
df['ad_id'] = df['ad_id'].astype(str)
df['price_per_m2'] = pd.to_numeric(df['price_per_m2'], errors='coerce')
df['area'] = pd.to_numeric(df['area'], errors='coerce')
df = df[(df['price_per_m2'] > 0) & (df['area'] > 0)].copy()
df['target_log_ppm2'] = np.log(df['price_per_m2'])
print('rows:', len(df))


In [ ]:
ad_ids = np.load(PROC_DIR / 'clip_vitb32_ad_ids.npy')
ad_emb = np.load(PROC_DIR / 'clip_vitb32_ad_emb.npy')
print('emb shape:', ad_emb.shape)
emb_cols = [f'clip_{i:03d}' for i in range(ad_emb.shape[1])]
emb_df = pd.DataFrame(ad_emb, columns=emb_cols)
emb_df['ad_id'] = ad_ids.astype(str)
df_feat = df.merge(emb_df, on='ad_id', how='inner')
print('merged rows:', len(df_feat))


In [ ]:
def norm_cat(v):
    if v is None:
        return 'unknown'
    s = str(v).strip()
    if not s or s.lower() in {'nan','none','null'}:
        return 'unknown'
    return s

df_feat['rooms'] = pd.to_numeric(df_feat['rooms'], errors='coerce')
df_feat['floor'] = pd.to_numeric(df_feat['floor'], errors='coerce')
df_feat['floors_total'] = pd.to_numeric(df_feat['floors_total'], errors='coerce')
df_feat['year_built'] = pd.to_numeric(df_feat['year_built'], errors='coerce')
df_feat['latitude'] = pd.to_numeric(df_feat['latitude'], errors='coerce')
df_feat['longitude'] = pd.to_numeric(df_feat['longitude'], errors='coerce')
df_feat['condition_confidence'] = pd.to_numeric(df_feat.get('condition_confidence', 0.0), errors='coerce').fillna(0.0)
df_feat['floor_ratio'] = np.where(df_feat['floors_total'] > 0, df_feat['floor'] / df_feat['floors_total'], 0.0)
df_feat['floor_ratio'] = df_feat['floor_ratio'].clip(0, 1)
df_feat['is_first'] = (df_feat['floor'] == 1).astype(int)
df_feat['is_last'] = ((df_feat['floors_total'] > 0) & (df_feat['floor'] == df_feat['floors_total'])).astype(int)
df_feat['building_age'] = 2026 - df_feat['year_built']
df_feat['has_geo'] = ((df_feat['latitude'].notna()) & (df_feat['longitude'].notna())).astype(int)
df_feat['has_residential_complex'] = df_feat['residential_complex'].fillna('').astype(str).str.strip().ne('').astype(int)
df_feat['latitude'] = df_feat['latitude'].fillna(43.238949)
df_feat['longitude'] = df_feat['longitude'].fillna(76.889709)

for c in ['district','building_type','residential_complex','object_type','condition_norm','condition_source']:
    if c not in df_feat.columns:
        df_feat[c] = 'unknown'
    df_feat[c] = df_feat[c].apply(norm_cat)

FEATURES_NUM = [
    'area','rooms','floor','floors_total','floor_ratio','is_first','is_last','building_age',
    'latitude','longitude','has_geo','has_residential_complex','condition_confidence'
]
FEATURES_CAT = ['district','building_type','residential_complex','object_type','condition_norm','condition_source']
FEATURES = FEATURES_NUM + FEATURES_CAT + emb_cols
df_feat = df_feat.dropna(subset=FEATURES_NUM + ['target_log_ppm2']).copy()
print('feature rows:', len(df_feat), 'features:', len(FEATURES))


In [ ]:
train_ids = set(pd.read_csv(PROC_DIR / 'train_ad_ids.csv', header=None).iloc[:,0].astype(str))
val_ids = set(pd.read_csv(PROC_DIR / 'val_ad_ids.csv', header=None).iloc[:,0].astype(str))
test_ids = set(pd.read_csv(PROC_DIR / 'test_ad_ids.csv', header=None).iloc[:,0].astype(str))

train_df = df_feat[df_feat['ad_id'].isin(train_ids)].copy()
val_df = df_feat[df_feat['ad_id'].isin(val_ids)].copy()
test_df = df_feat[df_feat['ad_id'].isin(test_ids)].copy()
print('splits:', len(train_df), len(val_df), len(test_df))
assert len(train_df) > 0 and len(val_df) > 0 and len(test_df) > 0


In [ ]:
GPU_AVAILABLE = True
try:
    import catboost
except Exception:
    GPU_AVAILABLE = False

model_kwargs = dict(
    loss_function='RMSE',
    eval_metric='RMSE',
    depth=8,
    learning_rate=0.05,
    n_estimators=1500,
    random_seed=42,
    verbose=False,
)
if GPU_AVAILABLE:
    model_kwargs.update(dict(task_type='GPU', devices='0'))

model = CatBoostRegressor(**model_kwargs)
print('CatBoost params:', model.get_params())
model.fit(
    train_df[FEATURES], train_df['target_log_ppm2'],
    cat_features=FEATURES_CAT,
    eval_set=(val_df[FEATURES], val_df['target_log_ppm2']),
    use_best_model=True,
)

def metrics(part):
    pred = model.predict(part[FEATURES])
    true_ppm2 = np.exp(part['target_log_ppm2'].to_numpy(float))
    pred_ppm2 = np.exp(np.asarray(pred, dtype=float))
    area = part['area'].to_numpy(float)
    true_price = true_ppm2 * area
    pred_price = pred_ppm2 * area
    mae_ppm2 = float(np.mean(np.abs(true_ppm2 - pred_ppm2)))
    rmse_ppm2 = float(np.sqrt(np.mean((true_ppm2 - pred_ppm2) ** 2)))
    mae_price = float(np.mean(np.abs(true_price - pred_price)))
    rmse_price = float(np.sqrt(np.mean((true_price - pred_price) ** 2)))
    return {'mae_ppm2': mae_ppm2, 'rmse_ppm2': rmse_ppm2, 'mae_price': mae_price, 'rmse_price': rmse_price}

metrics_val = metrics(val_df)
metrics_test = metrics(test_df)
print('VAL:', metrics_val)
print('TEST:', metrics_test)


In [ ]:
model_path = OUT_DIR / 'catboost_tabular_plus_clip_vitb32.cbm'
model.save_model(str(model_path))
print('Saved:', model_path)

v2_meta = {
  'version': 'v2_tabular_plus_clip_vitb32_mean10_time_split',
  'current_year': 2026,
  'target': 'log_price_per_m2',
  'features_num': FEATURES_NUM,
  'features_cat': FEATURES_CAT,
  'split_strategy': 'time_based_collected_at',
  'metrics_val': metrics_val,
  'metrics_test': metrics_test,
  'clip': {
    'library': 'open_clip',
    'model_name': 'ViT-B-32',
    'pretrained': 'laion2b_s34b_b79k',
    'aggregation': 'mean',
    'max_images': 10,
    'normalize_per_image': True,
    'normalize_agg': True
  },
  'model_file': 'catboost_tabular_plus_clip_vitb32.cbm'
}
(OUT_DIR / 'v2_metadata.json').write_text(json.dumps(v2_meta, ensure_ascii=False, indent=2), encoding='utf-8')
print('saved:', OUT_DIR / 'v2_metadata.json')
